In [3]:
# validation_df is the df we will use for the validation 

In [180]:
def section_validation(df, model, features):
    step = 100
    performances = []
    tot_samples = len(df)
    
    for start in range(0, tot_samples, step):
        end = min(start + step, tot_samples)  
        
        section_df = df.iloc[start:end]
        
        X_section = section_df[features]
        y_section = section_df['Trades']
    
        #step 1 predict with xgb
        y_pred = model.predict(X_section)
        precision = precision_score(y_section, y_pred, pos_label = 1)
        y_proba = model.predict_proba(X_section)[:, 1]
 
        # Calculate metrics for this section only
        precision_section = precision_score(y_section, y_pred, pos_label=1)
        recall_section = recall_score(y_section, y_pred, pos_label=1)
        f1_values = f1_score(y_section, y_pred, average = 'macro')
        
        performances.append({
            'section': f"{start}-{end}",
            'precision': precision_section,
            'recall': recall_section,
            'f1_values': f1_values,
            'trades_in_section': y_section.sum(),
            'signals_generated': y_pred.sum(),
            'section_size': len(section_df)
        })

    performances_df = pd.DataFrame(performances)
        
    return performances_df



In [182]:
def meta_validation(validation_df, model_a, model_b, model_c, model_meta, features_a, features_b, features_c):
    
    y_test = validation_df['Trades']   

    X_test_a = validation_df[features_a]
    X_test_b = validation_df[features_b]
    X_test_c = validation_df[features_c]
    
    proba_a = model_a.predict_proba(X_test_a)[:, 1]
    proba_b = model_b.predict_proba(X_test_b)[:, 1]
    proba_c = model_c.predict_proba(X_test_c)[:, 1]
    
    X_meta_test = np.column_stack([proba_a, proba_b, proba_c])

    y_pred_meta = model_meta.predict(X_meta_test)

    precision = precision_score(y_test, y_pred_meta, pos_label = 1)
    
    recall = recall_score(y_test, y_pred_meta, pos_label = 1)
    
    print(f'final model precision is: {precision} with a recall of: {recall}')

    return

In [186]:
def meta_section_validation(validation_df, model_a, model_b, model_c, model_meta, features_a, features_b, features_c):
    step = 100
    performances = []
    tot_samples = len(validation_df)
    
    for start in range(0, tot_samples, step):
        end = min(start + step, tot_samples)  
        
        section_df = validation_df.iloc[start:end]
        
        X_section_a = section_df[features_a]
        X_section_b = section_df[features_b]
        X_section_c = section_df[features_c]
        y_section = section_df['Trades']
    
        #step 1 predict with model_a
        proba_a = model_a.predict_proba(X_section_a)[:, 1]
        #step 2 predict with model_b
        proba_b = model_b.predict_proba(X_section_b)[:, 1]
        #step 3 predict with model_c
        proba_c = model_c.predict_proba(X_section_c)[:, 1]
        
        X_section_test = np.column_stack([proba_a, proba_b, proba_c])

        y_pred_meta = model_meta.predict(X_section_test)


         # Calculate metrics for this section only
        precision_section = precision_score(y_section, y_pred_meta, pos_label=1)
        recall_section = recall_score(y_section, y_pred_meta, pos_label=1)
        
        performances.append({
            'section': f"{start}-{end}",
            'precision': precision_section,
            'recall': recall_section,
            'trades_in_section': y_section.sum(),
            'signals_generated': y_pred_meta.sum(),
            'section_size': len(section_df)
        })

    performances_df = pd.DataFrame(performances)

        
    return performances_df
